In [1]:
from transformers import BertTokenizer, BertForSequenceClassification
import torch
import os
from google.colab import drive
drive.mount('/content/drive')

# Initialize the BERT tokenizer and model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertForSequenceClassification.from_pretrained('bert-base-uncased')

# Define a function to load data from the text files
def load_data(directory):
    data = []
    labels = []
    for label in ['pos', 'neg']:
        dir_name = os.path.join(directory, label)
        for fname in os.listdir(dir_name):
            if fname[-4:] == '.txt':
                f = open(os.path.join(dir_name, fname))
                data.append(f.read())
                f.close()
                if label == 'neg':
                    labels.append(0)
                else:
                    labels.append(1)
    return data, labels



Mounted at /content/drive


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [4]:
# Load training and testing data
#train_texts, train_labels = load_data('/content/drive/MyDrive/aclImdb/train')
#test_texts, test_labels = load_data('/content/drive/MyDrive/aclImdb/test')

train_texts, train_labels = load_data('aclImdb/train')
test_texts, test_labels = load_data('aclImdb/test')

# Preprocess the data
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=512)
test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=512)

# Convert to PyTorch DataLoaders
class IMDbDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = IMDbDataset(train_encodings, train_labels)
test_dataset = IMDbDataset(test_encodings, test_labels)



In [5]:
# Define the training arguments
training_args = {
    "epochs": 2,
    "batch_size": 16
}

# Train the model
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)
model.train()

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=training_args["batch_size"], shuffle=True)

optim = torch.optim.Adam(model.parameters(), lr=5e-5)

for epoch in range(training_args["epochs"]):
    for batch in train_loader:
        optim.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs[0]
        loss.backward()
        optim.step()
        print(f"Epoch: {epoch + 1}, Loss: {loss.item()}") # add this line



Epoch: 1, Loss: 0.7121515274047852
Epoch: 1, Loss: 0.715965986251831
Epoch: 1, Loss: 0.7313375473022461
Epoch: 1, Loss: 0.646247148513794
Epoch: 1, Loss: 0.732204020023346
Epoch: 1, Loss: 0.6956542730331421
Epoch: 1, Loss: 0.6776849627494812
Epoch: 1, Loss: 0.5760180950164795
Epoch: 1, Loss: 0.8051391839981079
Epoch: 1, Loss: 0.7267628312110901
Epoch: 1, Loss: 0.6498395204544067
Epoch: 1, Loss: 0.7205840945243835
Epoch: 1, Loss: 0.7098519206047058
Epoch: 1, Loss: 0.6515241861343384
Epoch: 1, Loss: 0.591627836227417
Epoch: 1, Loss: 0.624613881111145
Epoch: 1, Loss: 0.575435996055603
Epoch: 1, Loss: 0.49197685718536377
Epoch: 1, Loss: 0.5226454138755798
Epoch: 1, Loss: 0.37972670793533325
Epoch: 1, Loss: 0.47118663787841797
Epoch: 1, Loss: 0.41573914885520935
Epoch: 1, Loss: 0.2682325839996338
Epoch: 1, Loss: 0.650808572769165
Epoch: 1, Loss: 0.6061485409736633
Epoch: 1, Loss: 0.37473639845848083
Epoch: 1, Loss: 0.8580436706542969
Epoch: 1, Loss: 0.3540818691253662
Epoch: 1, Loss: 0.2280

In [6]:
model.eval()

# Evaluate the model
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=training_args["batch_size"], shuffle=True)

total = 0
correct = 0

for batch in test_loader:
    with torch.no_grad():
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        _, predicted = torch.max(outputs.logits, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print('Test Accuracy: ', 100 * correct / total)

Test Accuracy:  92.88


In [7]:
model.save_pretrained('bert_model.model')

In [8]:
def classify_review(review):
    inputs = tokenizer(review, padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        predicted_class = torch.argmax(outputs.logits, dim=1).item()
    return "Positive" if predicted_class == 1 else "Negative"


# Example usage
review = "This movie was fantastic! I loved every minute of it."
classification = classify_review(review)
print(f"Review: {review}")
print(f"Classification: {classification}")

review = "This movie was terrible. I hated it."
classification = classify_review(review)
print(f"Review: {review}")
print(f"Classification: {classification}")

Review: This movie was fantastic! I loved every minute of it.
Classification: Positive
Review: This movie was terrible. I hated it.
Classification: Negative
